In [ ]:


# 1. Selekcja danych i funkcje wierszowe

def pobierz_wizytowki_klientow(conn):
    """
    Generuje ustandaryzowane wizytówki klientów na potrzeby wysyłek pocztowych.
    
    Wykorzystuje selekcję danych oraz wbudowane funkcje wierszowe do modyfikacji 
    ciągów znaków. Łączy imię i nazwisko w jedno pole drukowanymi literami 
    oraz zabezpiecza braki w numerach telefonów za pomocą funkcji COALESCE.
    
    :param conn: Obiekt aktywnego połączenia z bazą danych (np. SQLAlchemy Engine).
    :type conn: sqlalchemy.engine.Engine
    :return: Ramka danych zawierająca sformatowane dane kontaktowe.
    :rtype: pandas.DataFrame
    """
    query = """
        SELECT 
            UPPER(imie || ' ' || nazwisko) AS klient_pelna_nazwa,
            COALESCE(telefon, 'BRAK KONTAKTU') AS telefon_kontaktowy,
            kod_pocztowy || ' ' || miasto AS adres_wysylkowy,
            LENGTH(email) AS dlugosc_emaila
        FROM 
            klienci
        ORDER BY 
            nazwisko;
    """
    return pd.read_sql_query(query, conn)



# 2. Funkcje agregujące

def oblicz_statystyki_sprzedazy(conn):
    """
    Oblicza sumaryczne statystyki dla każdego złożonego zamówienia.
    
    Wykorzystuje funkcje agregujące (SUM, COUNT) oraz klauzulę GROUP BY. 
    Oblicza całkowitą wartość finansową zamówienia na podstawie ilości i 
    ceny historycznej oraz zlicza unikalne pozycje w koszyku.
    
    :param conn: Obiekt aktywnego połączenia z bazą danych.
    :type conn: sqlalchemy.engine.Engine
    :return: Ramka danych ze statystykami finansowymi zamówień.
    :rtype: pandas.DataFrame
    """
    query = """
        SELECT 
            id_zamowienia,
            COUNT(id_produktu) AS liczba_unikalnych_produktow,
            SUM(ilosc) AS laczna_liczba_sztuk,
            SUM(ilosc * cena_historyczna) AS calkowita_wartosc_zamowienia
        FROM 
            szczegoly_zamowienia
        GROUP BY 
            id_zamowienia
        ORDER BY 
            calkowita_wartosc_zamowienia DESC;
    """
    return pd.read_sql_query(query, conn)



# 3. Połączenia - złączenia

def pobierz_pelny_paragon(conn, id_zamowienia):
    """
    Generuje szczegółowy rachunek (paragon) dla wskazanego zamówienia.
    
    Wykorzystuje wielokrotne złączenia (INNER JOIN) tabel: zamowienia, 
    klienci, szczegoly_zamowienia oraz produkty.
    Używa natywnej parametryzacji sterownika PostgreSQL %(nazwa)s.
    
    :param conn: Obiekt aktywnego połączenia z bazą danych.
    :type conn: sqlalchemy.engine.Engine
    :param id_zamowienia: Identyfikator zamówienia do wygenerowania rachunku.
    :type id_zamowienia: int
    :return: Ramka danych z pozycjami zamówienia i danymi kupującego.
    :rtype: pandas.DataFrame
    """
    query = """
        SELECT 
            z.id_zamowienia,
            z.data_zlozenia,
            k.imie || ' ' || k.nazwisko AS kupujacy,
            p.nazwa AS nazwa_produktu,
            sz.ilosc,
            sz.cena_historyczna
        FROM 
            zamowienia z
        JOIN 
            klienci k ON z.id_klienta = k.id_klienta
        JOIN 
            szczegoly_zamowienia sz ON z.id_zamowienia = sz.id_zamowienia
        JOIN 
            produkty p ON sz.id_produktu = p.id_produktu
        WHERE 
            z.id_zamowienia = %(id_zam)s;
    """
    return pd.read_sql_query(query, conn, params={"id_zam": id_zamowienia})


# 4. Operatory zbiorowe

def znajdz_produkty_bez_zamowien(conn):
    """
    Identyfikuje "martwy" asortyment, czyli produkty, które nigdy nie zostały kupione.
    
    Wykorzystuje operator różnicy zbiorów (EXCEPT). Odejmuje zbiór produktów 
    znajdujących się w koszykach (szczegolach_zamowienia) od zbioru wszystkich 
    produktów dostępnych w ofercie (tabela produkty).
    
    :param conn: Obiekt aktywnego połączenia z bazą danych.
    :type conn: sqlalchemy.engine.Engine
    :return: Ramka danych z listą produktów bez historii sprzedaży.
    :rtype: pandas.DataFrame
    """
    query = """
        SELECT 
            id_produktu, 
            nazwa 
        FROM 
            produkty
            
        EXCEPT
        
        SELECT 
            p.id_produktu, 
            p.nazwa 
        FROM 
            produkty p
        JOIN 
            szczegoly_zamowienia sz ON p.id_produktu = sz.id_produktu;
    """
    return pd.read_sql_query(query, conn)



# 5. Podzapytania

def znajdz_asortyment_premium(conn):
    """
    Wyszukuje produkty sklasyfikowane jako asortyment premium.
    
    Wykorzystuje podzapytanie skalarne w klauzuli WHERE. Produkt uznawany 
    jest za premium, jeśli jego cena bazowa jest wyższa niż średnia cena 
    wszystkich dostępnych w sklepie produktów.
    
    :param conn: Obiekt aktywnego połączenia z bazą danych.
    :type conn: sqlalchemy.engine.Engine
    :return: Ramka danych z produktami droższymi niż rynkowa średnia.
    :rtype: pandas.DataFrame
    """
    query = """
        SELECT 
            nazwa, 
            cena_bazowa 
        FROM 
            produkty
        WHERE 
            cena_bazowa > (
                SELECT AVG(cena_bazowa) 
                FROM produkty
            )
        ORDER BY 
            cena_bazowa DESC;
    """
    return pd.read_sql_query(query, conn)